In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path
import joblib
import os
import numpy as np

In [2]:
df = pd.read_csv("../data/processed/final_dataset2.csv")
df.head()

,Datetime,StationId,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,...,NO2_roll6,NO2_roll24,CO_roll3,CO_roll6,CO_roll24,O3_roll3,O3_roll6,O3_roll24,PM2.5_lag1,PM2.5_lag3
0,2017-11-25 12:00:00,AP001,79.00,124.00,5.30,21.15,15.53,9.40,0.1,15.40,...,18.807500,18.807500,0.1,0.1,0.1,147.650000,140.142500,140.142500,82.75,104.00
1,2017-11-25 13:00:00,AP001,79.00,124.00,5.30,21.15,15.53,9.40,0.1,15.40,...,19.276000,19.276000,0.1,0.1,0.1,154.506667,143.474000,143.474000,79.00,94.50
2,2017-11-25 14:00:00,AP001,68.50,117.00,1.35,13.60,8.35,7.40,0.1,21.80,...,18.330000,18.330000,0.1,0.1,0.1,158.433333,146.511667,146.511667,79.00,82.75
3,2017-11-25 15:00:00,AP001,69.25,112.25,1.52,11.80,7.55,9.25,0.1,21.38,...,16.463333,17.397143,0.1,0.1,0.1,160.060000,153.855000,148.678571,68.50,79.00
4,2017-11-25 16:00:00,AP001,70.00,107.00,2.80,30.33,18.40,6.15,0.1,18.90,...,18.810000,19.013750,0.1,0.1,0.1,157.116667,155.811667,148.590000,69.25,79.00


## Data Split

In [3]:
df["Datetime"] = pd.to_datetime(df["Datetime"])
df = (
    df.sort_values(["StationId", "Datetime"])
      .set_index("Datetime")
)

In [4]:
train_list, val_list, test_list = [], [], []

for station, group in df.groupby("StationId"):
    group = group.sort_index()
    n = len(group)

    # Minimal 100 data agar stasiun layak diikutkan
    if n < 100:
        continue

    train_end = int(n * 0.75)
    val_end   = int(n * 0.90)

    train_list.append(group.iloc[:train_end])
    val_list.append(group.iloc[train_end:val_end])
    test_list.append(group.iloc[val_end:])

train_df = pd.concat(train_list)
val_df   = pd.concat(val_list)
test_df  = pd.concat(test_list)

print(f"Train      : {len(train_df):,}")
print(f"Validation : {len(val_df):,}")
print(f"Test       : {len(test_df):,}")

Train      : 1,513,888
Validation : 302,770
Test       : 201,908


In [5]:
feature_cols_lstm = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'CO', 'SO2', 'O3', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'is_weekend', 'PM2.5_roll3', 'PM2.5_roll6', 'PM2.5_roll24', 'PM10_roll3', 'PM10_roll6', 'PM10_roll24', 'NO2_roll3', 'NO2_roll6', 'NO2_roll24', 'CO_roll3', 'CO_roll6', 'CO_roll24', 'O3_roll3', 'O3_roll6', 'O3_roll24', 'PM2.5_lag1', 'PM2.5_lag3']
target_col = "AQI"

## Min Max Scaler

In [6]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

In [7]:
# Fit hanya pada train, transform semua set
train_df = train_df.copy()
val_df   = val_df.copy()
test_df  = test_df.copy()

# Normalisasi fitur
train_df[feature_cols_lstm] = scaler_X.fit_transform(
    train_df[feature_cols_lstm])
val_df[feature_cols_lstm] = scaler_X.transform(
    val_df[feature_cols_lstm])
test_df[feature_cols_lstm] = scaler_X.transform(
    test_df[feature_cols_lstm])

# Normalisasi target AQI
train_df[[target_col]] = scaler_y.fit_transform(
    train_df[[target_col]])
val_df[[target_col]] = scaler_y.transform(
    val_df[[target_col]])
test_df[[target_col]] = scaler_y.transform(
    test_df[[target_col]])

print("Cek range fitur (train):")
print(
    train_df[feature_cols_lstm]
    .describe()
    .loc[["min", "max"]]
)

print("\nCek range target (train):")
print(
    train_df[[target_col]]
    .describe()
    .loc[["min", "max"]]
)

Cek range fitur (train):
     PM2.5  PM10   NO  NO2  NOx   CO  SO2   O3  hour_sin  hour_cos  ...  \
min    0.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0       0.0       0.0  ...   
max    1.0   1.0  1.0  1.0  1.0  1.0  1.0  1.0       1.0       1.0  ...   

     NO2_roll6  NO2_roll24  CO_roll3  CO_roll6  CO_roll24  O3_roll3  O3_roll6  \
min        0.0         0.0       0.0       0.0        0.0       0.0       0.0   
max        1.0         1.0       1.0       1.0        1.0       1.0       1.0   

     O3_roll24  PM2.5_lag1  PM2.5_lag3  
min        0.0         0.0         0.0  
max        1.0         1.0         1.0  

[2 rows x 30 columns]

Cek range target (train):
     AQI
min  0.0
max  1.0


In [8]:
# Membuat folder artifacts jika belum ada
os.makedirs("../artifacts", exist_ok=True)

# Simpan scaler
joblib.dump(scaler_X, "../artifacts/2scaler_X.pkl")
joblib.dump(scaler_y, "../artifacts/2scaler_y.pkl")

print("Scaler berhasil disimpan.")

Scaler berhasil disimpan.


## Sliding Window

In [9]:
print(len(train_df))
print(len(val_df))
print(len(test_df))

1513888
302770
201908


In [10]:
def create_sequences(df, feature_cols, target_col="AQI", window_size=24):
    X, y = [], []

    for station, group in df.groupby("StationId"):
        group = group.sort_index()
        timestamps = group.index.to_series()

        features = group[feature_cols].values
        targets  = group[target_col].values

        for i in range(len(group) - window_size):
            # Cek kontinuitas window (tidak ada gap)
            time_diff = timestamps.iloc[i:i + window_size + 1].diff().dropna()
            if not (time_diff == pd.Timedelta(hours=1)).all():
                continue

            X.append(features[i:i + window_size])
            y.append(targets[i + window_size])

    return np.array(X), np.array(y)

WINDOW_SIZE = 24

X_train, y_train = create_sequences(train_df, feature_cols_lstm, target_col, WINDOW_SIZE)
X_val,   y_val   = create_sequences(val_df,   feature_cols_lstm, target_col, WINDOW_SIZE)
X_test,  y_test  = create_sequences(test_df,  feature_cols_lstm, target_col, WINDOW_SIZE)

print(f"X_train : {X_train.shape}  |  y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}    |  y_val   : {y_val.shape}")
print(f"X_test  : {X_test.shape}   |  y_test  : {y_test.shape}")

X_train : (1440581, 24, 30)  |  y_train : (1440581,)
X_val   : (289676, 24, 30)    |  y_val   : (289676,)
X_test  : (191295, 24, 30)   |  y_test  : (191295,)


## Simpan Data Train

In [11]:
# Membuat folder artifacts jika belum ada
os.makedirs("../artifacts", exist_ok=True)

# Simpan seluruh sequence dalam satu file
np.savez_compressed(
    "../artifacts/dataset_sequences2.npz",
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    X_test=X_test,
    y_test=y_test
)

print("Sliding window dataset berhasil disimpan.")

Sliding window dataset berhasil disimpan.


In [12]:
os.makedirs("../artifacts", exist_ok=True)

# Metadata dataset
metadata = {
    "window_size": WINDOW_SIZE,
    "feature_cols": feature_cols_lstm,
    "target_col": target_col,
    "n_features": len(feature_cols_lstm),
    "input_shape": X_train.shape[1:] if len(X_train) > 0 else (WINDOW_SIZE, len(feature_cols_lstm))
}

# Simpan metadata
joblib.dump(metadata, "../artifacts/dataset_metadata2.pkl")

print("Metadata dataset berhasil disimpan.")

Metadata dataset berhasil disimpan.
